In [32]:
import numpy as np, itertools
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.integrate import quad

In [45]:
lam = np.array([3, 2, 1.5, 1.2, 1, 0.75])
n = len(lam)
rng = np.random.default_rng(0)


N = 2 ** n
rate = np.zeros(N)
for S in range(1, N):
    rate[S] = sum(lam[i] for i in range(n) if S >> i & 1)
EM = np.zeros(N)
plast = np.zeros((N, n))
for S in range(1, N):
    members = [i for i in range(n) if S >> i & 1]
    if len(members) == 1:
        EM[S] = 1 / lam[members[0]]
        plast[S, members[0]] = 1
        continue
    EM[S] = 1 / rate[S]
    for i in members:
        T = S & ~(1 << i)
        w = lam[i] / rate[S]
        EM[S] += w * EM[T]
        plast[S] += w * plast[T]
full = N - 1
print("E[M] =", EM[full])
print("P(last):", np.round(plast[full], 5))


alt = 0.0
for r in range(1, n + 1):
    for S in itertools.combinations(range(n), r):
        alt += (-1) ** (r + 1) / lam[list(S)].sum()
print("Alternating E[M] =", alt)

E[M] = 2.1073764063261615
P(last): [0.02679 0.06953 0.12299 0.18125 0.24098 0.35846]
Alternating E[M] = 2.107376406326162


In [44]:
R = 200_000
X = rng.exponential(1 / lam, size=(R, n))
M = X.max(axis=1)
last = X.argmax(axis=1)
print("Simulated E[M] =", M.mean(), "+/-", M.std() / np.sqrt(R))
print("Simulated P(last):", np.round(np.bincount(last, minlength=n) / R, 5))

def dens(t):
    t = np.asarray(t, float)
    tot = 0
    for i in range(n):
        term = lam[i] * np.exp(-lam[i] * t)
        for j in range(n):
            if j != i:
                term = term * (1 - np.exp(-lam[j] * t))
        tot = tot + term
    return tot
print("Exact density integrates to", quad(dens, 0, np.inf)[0],
      "mean from density", quad(lambda t: t * dens(t), 0, np.inf)[0])
plt.figure(figsize=(7, 4))
plt.hist(M, bins=80, density=True, alpha=0.6, label="simulated M")
tt = np.linspace(0, M.max(), 600)
plt.plot(tt, dens(tt), "r", lw=2, label="exact density")
plt.xlabel("t")
plt.ylabel("density"); plt.legend(); plt.title("Finishing time of last student")
plt.tight_layout()
plt.savefig("hist_M.png", dpi=150)

Simulated E[M] = 2.1131605629093557 +/- 0.0028083390046551387
Simulated P(last): [0.02742 0.06942 0.12345 0.18086 0.24024 0.3586 ]
Exact density integrates to 0.9999999999999972 mean from density 2.107376406326073


In [46]:
X1 = X.min(axis=1)
I = X.argmin(axis=1)
print("\nFirst finish: Lambda =", lam.sum(), " 1/Lambda =", 1 / lam.sum())
for i in range(n):
    s = X1[I == i]
    print(f"  I={i+1}: P={len(s)/R:.4f} (theory {lam[i]/lam.sum():.4f}), "
          f"mean X={s.mean():.4f}, sd={s.std():.4f}, P(X>0.1)={np.mean(s>0.1):.4f}")
print("  theory mean 1/Lambda =", 1 / lam.sum(), " P(X>0.1) =", np.exp(-0.1 * lam.sum()))


First finish: Lambda = 9.45  1/Lambda = 0.10582010582010583
  I=1: P=0.3173 (theory 0.3175), mean X=0.1054, sd=0.1059, P(X>0.1)=0.3862
  I=2: P=0.2112 (theory 0.2116), mean X=0.1068, sd=0.1069, P(X>0.1)=0.3910
  I=3: P=0.1605 (theory 0.1587), mean X=0.1065, sd=0.1075, P(X>0.1)=0.3901
  I=4: P=0.1252 (theory 0.1270), mean X=0.1059, sd=0.1055, P(X>0.1)=0.3904
  I=5: P=0.1062 (theory 0.1058), mean X=0.1058, sd=0.1054, P(X>0.1)=0.3894
  I=6: P=0.0795 (theory 0.0794), mean X=0.1072, sd=0.1067, P(X>0.1)=0.3939
  theory mean 1/Lambda = 0.10582010582010583  P(X>0.1) = 0.38867957090175304


In [48]:
def EM_rates(k):
    f = lambda t: 1 - np.prod(1 - np.exp(-np.arange(1, k + 1) * t))
    return quad(f, 0, np.inf, limit=200)[0]
ks = np.arange(1, 13)
vals = [EM_rates(k) for k in ks]

def EM_dp(k):
    NN = 1 << k
    r = np.zeros(NN); E = np.zeros(NN)
    for S in range(1, NN):
        mem = [i + 1 for i in range(k) if S >> i & 1]
        r[S] = sum(mem)
        E[S] = 1 / r[S] + sum(i / r[S] * E[S & ~(1 << (i - 1))] for i in mem)
    return E[NN - 1]

plt.figure(figsize=(7, 4))
plt.plot(ks, vals, "o-", label="E[M]")
plt.axhline(2 * np.log(2), color="r", ls="--", label="2 ln 2")
plt.xlabel("k (rates 1,...,k)"); plt.ylabel("E[M]"); plt.legend()
plt.title("E[M] with rates 1,...,k"); plt.tight_layout()
plt.savefig("EM_vs_k.png", dpi=150); plt.close()